In [ ]:
############################################
# INSTALL LIBRARIES
############################################

!pip install kaggle
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install transformers sentence-transformers scikit-learn pandas numpy openai gradio


Looking in indexes: https://download.pytorch.org/whl/cpu


In [ ]:
############################################
# KAGGLE API SETUP (UPLOAD kaggle.json)
############################################
from google.colab import files

print("➡️ Upload your kaggle.json file (from Kaggle account)")
uploaded = files.upload()

import os, json

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open("kaggle.json", "rb") as f_in, open(os.path.expanduser("~/.kaggle/kaggle.json"), "wb") as f_out:
    f_out.write(f_in.read())

os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

print("✅ Kaggle API configured.")


➡️ Upload your kaggle.json file (from Kaggle account)


Saving kaggle.json to kaggle.json
✅ Kaggle API configured.


In [ ]:
############################################
# DOWNLOAD "Customer Support on Twitter" DATASET
############################################

!kaggle datasets download -d thoughtvector/customer-support-on-twitter
!unzip -o customer-support-on-twitter.zip

import pandas as pd

df_raw = pd.read_csv("/content/twcs/twcs.csv")
df_raw.head()

Dataset URL: https://www.kaggle.com/datasets/thoughtvector/customer-support-on-twitter
License(s): CC-BY-NC-SA-4.0
customer-support-on-twitter.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  customer-support-on-twitter.zip
  inflating: sample.csv              
  inflating: twcs/twcs.csv           


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [ ]:
############################################
# BUILD A 2000-ROW DATASET WITH CUSTOM INTENTS
############################################

import re

def map_intent(text: str) -> str:
    t = str(text).lower()
    if "refund" in t or "money back" in t:
        return "refund_request"
    if "password" in t or "login" in t or "log in" in t or "sign in" in t:
        return "account_issue"
    if "where is" in t or "delayed" in t or "late" in t or "when will" in t:
        return "order_status"
    if "cancel" in t or "unsubscribe" in t or "stop subscription" in t:
        return "cancel_subscription"
    if "charge" in t or "billing" in t or "charged" in t or "invoice" in t:
        return "billing_issue"
    if "app" in t or "site" in t or "website" in t or "error" in t or "crash" in t:
        return "technical_issue"
    if "broken" in t or "damaged" in t or "faulty" in t or "defective" in t:
        return "product_issue"
    if "card" in t or "transaction" in t or "declined" in t or "payment" in t:
        return "payment_issue"
    if "promo" in t or "coupon" in t or "discount" in t or "offer" in t:
        return "promo_issue"
    if "return" in t or "send back" in t:
        return "return_request"
    if "subscription" in t or "membership" in t:
        return "subscription_issue"
    if "complain" in t or "unhappy" in t or "terrible" in t or "worst" in t:
        return "complaint"
    return "general_query"


df_inbound = df_raw[df_raw["inbound"] == True].copy()


df_sample = df_inbound.sample(2000, random_state=42).copy()

df_sample["intent"] = df_sample["text"].apply(map_intent)
df_sample = df_sample[["text", "intent"]].dropna()

print(df_sample["intent"].value_counts())
df_sample.head()


intent
general_query          1368
technical_issue         302
order_status             75
refund_request           48
billing_issue            44
payment_issue            44
cancel_subscription      29
complaint                27
promo_issue              27
account_issue            20
return_request           10
product_issue             5
subscription_issue        1
Name: count, dtype: int64


,text,intent
26861,@AppleSupport Basically for a chat to be opene...,technical_issue
211386,@AppleSupport iOS 11.02 and Watchos4.0: No ico...,technical_issue
78521,"Dear god not again,@AppleSupport https://t.co/...",technical_issue
1225222,@ATVIAssist Hi there! If I buy Call of Duty WW...,general_query
194583,Hi @Safaricom_Care why can't I pay my my Dstv ...,general_query


In [ ]:
############################################
# PREPARE DATA FOR BERT TRAINING
############################################
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df = df_sample.copy()


intent_counts = df['intent'].value_counts()
rare_intents = intent_counts[intent_counts < 2].index
df = df[~df['intent'].isin(rare_intents)]

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["intent"])

X = df["text"].tolist()
y = df["label"].tolist()

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.10, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20, random_state=42, stratify=y_train_full
)

print("Train:", len(X_train), "Val:", len(X_val), "Test:", len(X_test))
print("Number of intents:", len(label_encoder.classes_))
print("Intents:", list(label_encoder.classes_))

Train: 1439 Val: 360 Test: 200
Number of intents: 12
Intents: ['account_issue', 'billing_issue', 'cancel_subscription', 'complaint', 'general_query', 'order_status', 'payment_issue', 'product_issue', 'promo_issue', 'refund_request', 'return_request', 'technical_issue']


In [ ]:
############################################
# DATASET CLASS & LOADERS
############################################
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }

BATCH_SIZE = 16

train_ds = IntentDataset(X_train, y_train, tokenizer)
val_ds   = IntentDataset(X_val,   y_val,   tokenizer)
test_ds  = IntentDataset(X_test,  y_test,  tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

len(train_loader), len(val_loader), len(test_loader)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

(90, 23, 13)

In [ ]:
############################################
# LOAD BERT MODEL (INTENT CLASSIFIER)
############################################
from transformers import BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

try:
    model = BertForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=len(label_encoder.classes_),
        ignore_mismatched_sizes=True
    )
except Exception as e:
    print("Error loading BertForSequenceClassification:", e)
    from transformers import AutoModelForSequenceClassification
    model = AutoModelForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=len(label_encoder.classes_),
        ignore_mismatched_sizes=True
    )

model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

total_steps = len(train_loader) * 3
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

print("✅ Model and optimizer ready.")

Using device: cpu


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model and optimizer ready.


In [ ]:
############################################
# TRAINING & EVALUATION FUNCTIONS
############################################
from sklearn.metrics import classification_report

def train_epoch(model, loader):
    model.train()
    total_loss = 0.0

    for batch in loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)

@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    preds, labels_all = [], []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        logits = outputs.logits
        batch_preds = torch.argmax(logits, dim=-1)

        preds.extend(batch_preds.cpu().numpy())
        labels_all.extend(labels.cpu().numpy())

    report = classification_report(
        labels_all, preds,
        target_names=label_encoder.classes_,
        labels=range(len(label_encoder.classes_)),
        digits=4
    )
    return report

In [ ]:
############################################
# TRAIN THE INTENT CLASSIFIER (3 EPOCHS)
############################################
EPOCHS = 3

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Train loss: {train_loss:.4f}")
    print("Validation performance:")
    print(eval_epoch(model, val_loader))


Epoch 1/3 - Train loss: 1.3435
Validation performance:


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                     precision    recall  f1-score   support

      account_issue     0.0000    0.0000    0.0000         4
      billing_issue     0.0000    0.0000    0.0000         8
cancel_subscription     0.0000    0.0000    0.0000         5
          complaint     0.0000    0.0000    0.0000         5
      general_query     0.7193    1.0000    0.8367       246
       order_status     0.0000    0.0000    0.0000        13
      payment_issue     0.0000    0.0000    0.0000         8
      product_issue     0.0000    0.0000    0.0000         1
        promo_issue     0.0000    0.0000    0.0000         5
     refund_request     0.0000    0.0000    0.0000         9
     return_request     0.0000    0.0000    0.0000         2
    technical_issue     0.8889    0.2963    0.4444        54

           accuracy                         0.7278       360
          macro avg     0.1340    0.1080    0.1068       360
       weighted avg     0.6249    0.7278    0.6384       360

Epoch 2/3 - Train los

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                     precision    recall  f1-score   support

      account_issue     0.0000    0.0000    0.0000         4
      billing_issue     0.0000    0.0000    0.0000         8
cancel_subscription     0.0000    0.0000    0.0000         5
          complaint     0.0000    0.0000    0.0000         5
      general_query     0.7531    0.9919    0.8561       246
       order_status     0.0000    0.0000    0.0000        13
      payment_issue     0.0000    0.0000    0.0000         8
      product_issue     0.0000    0.0000    0.0000         1
        promo_issue     0.0000    0.0000    0.0000         5
     refund_request     0.0000    0.0000    0.0000         9
     return_request     0.0000    0.0000    0.0000         2
    technical_issue     0.8611    0.5741    0.6889        54

           accuracy                         0.7639       360
          macro avg     0.1345    0.1305    0.1288       360
       weighted avg     0.6438    0.7639    0.6884       360

Epoch 3/3 - Train los

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
############################################
# TEST SET EVALUATION
############################################
print("Test set performance:")
print(eval_epoch(model, test_loader))


Test set performance:
                     precision    recall  f1-score   support

      account_issue     0.0000    0.0000    0.0000         2
      billing_issue     0.0000    0.0000    0.0000         4
cancel_subscription     0.0000    0.0000    0.0000         3
          complaint     0.0000    0.0000    0.0000         3
      general_query     0.7765    0.9635    0.8599       137
       order_status     0.0000    0.0000    0.0000         8
      payment_issue     0.0000    0.0000    0.0000         4
      product_issue     0.0000    0.0000    0.0000         0
        promo_issue     0.0000    0.0000    0.0000         3
     refund_request     0.0000    0.0000    0.0000         5
     return_request     0.0000    0.0000    0.0000         1
    technical_issue     0.7000    0.7000    0.7000        30

           accuracy                         0.7650       200
          macro avg     0.1230    0.1386    0.1300       200
       weighted avg     0.6369    0.7650    0.6941       200


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

In [ ]:
############################################
# SENTIMENT ANALYSIS AGENT
############################################
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

sent_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
sent_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english"
).to(device)
sent_model.eval()

def analyse_sentiment(text: str):
    enc = sent_tok(text, return_tensors="pt", truncation=True, padding=True)
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    with torch.no_grad():
        outputs = sent_model(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits, dim=-1)[0]
        pred = torch.argmax(probs).item()

    label = "positive" if pred == 1 else "negative"
    confidence = float(probs[pred])
    return label, confidence

analyse_sentiment("I am very unhappy with your service.")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

('negative', 0.9996771812438965)

In [ ]:
############################################
# FAQ KNOWLEDGE BASE (MANUAL, SIMPLE)
############################################

faq_data = {
    "intent": [
        "refund_request",
        "account_issue",
        "order_status",
        "cancel_subscription",
        "billing_issue",
        "technical_issue",
        "product_issue",
        "payment_issue",
        "promo_issue",
        "return_request",
        "subscription_issue",
        "complaint",
        "general_query"
    ],
    "question": [
        "How do I request a refund?",
        "I can't log in to my account.",
        "Where is my order?",
        "How do I cancel my subscription?",
        "Why was I charged incorrectly?",
        "The app or website is not working.",
        "The product I received is faulty.",
        "Why is my payment failing?",
        "Why is my promo code not working?",
        "How do I return an item?",
        "Why is my subscription not activating?",
        "I want to file a complaint.",
        "I have a general question."
    ],
    "answer": [
        "You can request a refund via your account's 'Orders' section and selecting 'Request refund'.",
        "Please use the 'Forgot Password' option on the login page to reset your credentials.",
        "You can track your order from the 'Orders' page in your account dashboard.",
        "You can cancel your subscription anytime in the 'Subscription' or 'Billing' settings page.",
        "Please check your billing statement; if the issue remains, contact billing support with your order ID.",
        "Try clearing your cache, restarting the app or browser, and ensure your internet connection is stable.",
        "We are sorry for the inconvenience. You can request a replacement or return from your orders page.",
        "Please verify your card details and ensure your bank has not blocked the transaction.",
        "Check that the promo code is valid, not expired, and meets the minimum order requirements.",
        "Go to the 'Orders' page, select the item, and choose the 'Return' option.",
        "Log out and log back in; if the issue continues, contact support with your subscription ID.",
        "We apologise for the poor experience. Please share details and we will escalate this to our team.",
        "You can find frequently asked questions in the Help Center or contact support with your query."
    ]
}

faq_df = pd.DataFrame(faq_data)
faq_df


,intent,question,answer
0,refund_request,How do I request a refund?,You can request a refund via your account's 'O...
1,account_issue,I can't log in to my account.,Please use the 'Forgot Password' option on the...
2,order_status,Where is my order?,You can track your order from the 'Orders' pag...
3,cancel_subscription,How do I cancel my subscription?,You can cancel your subscription anytime in th...
4,billing_issue,Why was I charged incorrectly?,Please check your billing statement; if the is...
5,technical_issue,The app or website is not working.,"Try clearing your cache, restarting the app or..."
6,product_issue,The product I received is faulty.,We are sorry for the inconvenience. You can re...
7,payment_issue,Why is my payment failing?,Please verify your card details and ensure you...
8,promo_issue,Why is my promo code not working?,"Check that the promo code is valid, not expire..."
9,return_request,How do I return an item?,"Go to the 'Orders' page, select the item, and ..."


In [ ]:
############################################
# FAQ RETRIEVAL AGENT (SENTENCE EMBEDDINGS)
############################################
from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer("all-MiniLM-L6-v2")

faq_questions = faq_df["question"].tolist()
faq_embeddings = embedder.encode(faq_questions, convert_to_tensor=True)

def retrieve_faq_answer(user_query: str):
    query_embedding = embedder.encode(user_query, convert_to_tensor=True)
    hits = util.semantic_search(query_embedding, faq_embeddings, top_k=1)[0]
    best = hits[0]
    idx = best["corpus_id"]
    score = float(best["score"])
    return {
        "question": faq_df.iloc[idx]["question"],
        "answer": faq_df.iloc[idx]["answer"],
        "similarity": score
    }

retrieve_faq_answer("I can't access my account")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

{'question': "I can't log in to my account.",
 'answer': "Please use the 'Forgot Password' option on the login page to reset your credentials.",
 'similarity': 0.8380143642425537}

In [ ]:
############################################
# DECISION AGENT
############################################
def decision_agent(intent_confidence: float, faq_similarity: float, sentiment_label: str):
    """
    Simple rule-based decision:
    - If intent prediction is confident AND FAQ match is strong AND sentiment is not negative => auto_resolve
    - Otherwise => escalate to human
    """
    if intent_confidence > 0.75 and faq_similarity > 0.60 and sentiment_label != "negative":
        return "auto_resolve"
    else:
        return "escalate"


decision_agent(0.92, 0.81, "positive")


'auto_resolve'

In [ ]:
############################################
# LOAD TRAINED MODEL FOR INFERENCE
############################################
from transformers import BertForSequenceClassification

inference_model = model
inference_model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
############################################
# INTENT PREDICTION FUNCTION
############################################
def predict_intent(text: str):
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    )
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    with torch.no_grad():
        outputs = inference_model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1)[0]
        pred_idx = torch.argmax(probs).item()

    intent_name = label_encoder.inverse_transform([pred_idx])[0]
    confidence = float(probs[pred_idx])
    return intent_name, confidence

predict_intent("I want a refund for my last booking.")


('general_query', 0.3440423309803009)

In [ ]:
############################################
# FULL MULTI-AGENT PIPELINE FUNCTION
############################################
def full_pipeline(user_input: str):

    intent, intent_conf = predict_intent(user_input)

    sentiment, sent_conf = analyse_sentiment(user_input)

    faq = retrieve_faq_answer(user_input)

    decision = decision_agent(intent_conf, faq["similarity"], sentiment)

    if decision == "auto_resolve":
        reply = faq["answer"]
    else:
        reply = "Your issue has been escalated to a human support agent. You will be contacted soon."

    return {
        "intent": intent,
        "intent_confidence": intent_conf,
        "sentiment": sentiment,
        "sentiment_confidence": sent_conf,
        "faq_question": faq["question"],
        "faq_answer": faq["answer"],
        "faq_similarity": faq["similarity"],
        "decision": decision,
        "final_reply": reply
    }

# Example
full_pipeline("I can't access my account.")


{'intent': 'general_query',
 'intent_confidence': 0.5374305248260498,
 'sentiment': 'negative',
 'sentiment_confidence': 0.9995786547660828,
 'faq_question': "I can't log in to my account.",
 'faq_answer': "Please use the 'Forgot Password' option on the login page to reset your credentials.",
 'faq_similarity': 0.8630715608596802,
 'decision': 'escalate',
 'final_reply': 'Your issue has been escalated to a human support agent. You will be contacted soon.'}

In [ ]:
############################################
# TYPE-YOUR-OWN-QUERY (MANUAL)
############################################
user_query = input("Enter your customer query: ")
result = full_pipeline(user_query)
result


Enter your customer query: i want a refund


{'intent': 'general_query',
 'intent_confidence': 0.4542334973812103,
 'sentiment': 'negative',
 'sentiment_confidence': 0.994002640247345,
 'faq_question': 'How do I request a refund?',
 'faq_answer': "You can request a refund via your account's 'Orders' section and selecting 'Request refund'.",
 'faq_similarity': 0.818439781665802,
 'decision': 'escalate',
 'final_reply': 'Your issue has been escalated to a human support agent. You will be contacted soon.'}

In [ ]:
############################################
# GRADIO UI (INTERACTIVE FRONTEND)
############################################
import gradio as gr

def chatbot(query):
    result = full_pipeline(query)
    return (
        f"Intent: {result['intent']} (conf {result['intent_confidence']:.2f})\n"
        f"Sentiment: {result['sentiment']} (conf {result['sentiment_confidence']:.2f})\n"
        f"FAQ match: {result['faq_question']}\n"
        f"Answer: {result['faq_answer']} (sim {result['faq_similarity']:.2f})\n"
        f"Decision: {result['decision']}\n\n"
        f"Final Reply: {result['final_reply']}"
    )

demo = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(lines=2, placeholder="Type your customer query…"),
    outputs=gr.Textbox(lines=15, max_lines=30, label="Agentic AI Response"),
    title="Agentic Customer Support AI (Kaggle – Twitter Data By Neha)",
    description="Type any customer support query to see how the multi-agent system responds."
)

demo.launch()



It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4a0d9d7e8ff5d61e5a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
